# 04. Genetic Programming — Symbolic Regression

Notebook minh họa **Genetic Programming (GP)** — một biến thể của GA
trong đó mỗi cá thể là một CÂY BIỂU THỨC (expression tree) thay vì một
vector số cố định như các notebook trước — áp dụng cho bài toán
**Symbolic Regression** (theo khung ứng dụng trong `README.md`).

**Symbolic Regression khác gì hồi quy thông thường?** Hồi quy thông
thường (tuyến tính, đa thức...) CỐ ĐỊNH trước dạng mô hình rồi chỉ đi
tìm THAM SỐ tốt nhất cho đúng dạng đó. Symbolic Regression không giả
định trước dạng hàm — GP tìm kiếm đồng thời cả CẤU TRÚC (những toán tử
nào, kết hợp với nhau ra sao) lẫn HẰNG SỐ của biểu thức, trực tiếp từ
dữ liệu.

**Bài toán mẫu**: có một hàm số $f(x)$ mà ta chỉ quan sát được qua dữ
liệu nhiễu $(x_i, y_i)$ — GP không được biết trước công thức này. Nhiệm
vụ: chỉ từ các cặp $(x_i, y_i)$, tìm lại một biểu thức toán học khớp
với dữ liệu — lý tưởng nhất là trùng với $f(x)$ thật.

---
##### Import package

In [1]:
import time

import numpy as np
import sympy as sp
from IPython.display import Markdown, display

print(f"numpy {np.__version__} | sympy {sp.__version__}")

numpy 1.26.4 | sympy 1.14.0


---
##### Sinh dữ liệu mẫu (ẩn công thức thật)

In [2]:
# Hàm ẩn dùng để SINH dữ liệu — GP không được biết công thức này, chỉ
# thấy các cặp (x, y) nhiễu bên dưới. Ta chỉ dùng lại nó ở cuối notebook
# để chấm điểm khách quan (so công thức GP tìm được với công thức thật).
def ham_an(x):
    return x**2 + np.sin(x)


N_TRAIN = 60
NOISE_STD = 0.1

rng = np.random.default_rng(42)
X_train = rng.uniform(-3, 3, size=N_TRAIN)
y_train = ham_an(X_train) + rng.normal(0, NOISE_STD, size=N_TRAIN)

print(f"Số điểm dữ liệu: {N_TRAIN}")
print("5 điểm đầu tiên (x, y):")
for x, y in zip(X_train[:5], y_train[:5]):
    print(f"  ({x:.4f}, {y:.4f})")

Số điểm dữ liệu: 60
5 điểm đầu tiên (x, y):
  (1.6437, 3.6126)
  (-0.3667, -0.1272)
  (2.1516, 5.2971)
  (1.1842, 2.2951)
  (-2.4349, 5.2959)


---
##### Biểu diễn cá thể — cây biểu thức (expression tree)

Mỗi cá thể là một cây, mỗi nút là:
- **Hàm** (arity 2: `+ - * /`, arity 1: `sin cos`) — có các nút con.
- **Biến** `x` — nút lá.
- **Hằng số** — một số thực ngẫu nhiên, cũng là nút lá.

`/` được "bảo vệ" (protected division): nếu kết quả không hữu hạn (chia
cho 0) thì trả về 1 thay vì lỗi/`inf` — quy ước kinh điển của Koza để
GP không bị crash khi sinh ra cây chứa phép chia cho 0.

In [3]:
FUNCTIONS_ARITY = {
    "+": 2, "-": 2, "*": 2, "/": 2, "sin": 1, "cos": 1,
}


def protected_div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        ket_qua = np.divide(a, b)
    return np.where(np.isfinite(ket_qua), ket_qua, 1.0)


def apply_op(op, args):
    if op == "+":
        return args[0] + args[1]
    if op == "-":
        return args[0] - args[1]
    if op == "*":
        return args[0] * args[1]
    if op == "/":
        return protected_div(args[0], args[1])
    if op == "sin":
        return np.sin(args[0])
    if op == "cos":
        return np.cos(args[0])
    raise ValueError(f"Toán tử không xác định: {op}")


class Node:
    """Một nút trong cây biểu thức. `op` là tên hàm ("+", "sin", ...),
    "x" (biến), hoặc một số thực (hằng số). `children` rỗng với nút lá."""

    __slots__ = ("op", "children")

    def __init__(self, op, children=None):
        self.op = op
        self.children = children if children is not None else []


def eval_node(node, x):
    """Tính giá trị cây tại (mảng) x — đệ quy từ lá lên gốc."""
    if node.op == "x":
        return x
    if isinstance(node.op, (int, float)):
        return node.op
    args = [eval_node(con, x) for con in node.children]
    return apply_op(node.op, args)


def tree_depth(node):
    if not node.children:
        return 1
    return 1 + max(tree_depth(con) for con in node.children)


def tree_size(node):
    return 1 + sum(tree_size(con) for con in node.children)


def copy_tree(node):
    return Node(node.op, [copy_tree(con) for con in node.children])


def all_nodes(node):
    """Liệt kê mọi nút trong cây — dùng để chọn điểm lai ghép/đột biến."""
    nodes = [node]
    for con in node.children:
        nodes.extend(all_nodes(con))
    return nodes

---
##### Khởi tạo quần thể — Ramped Half-and-Half

Phương pháp kinh điển của Koza: nửa quần thể sinh bằng **"full"** (mọi
nhánh đi đến đúng độ sâu quy định, cây "đầy"), nửa còn lại bằng
**"grow"** (mỗi nhánh có thể dừng sớm ở lá, cây hình dạng bất kỳ) —
đồng thời rải đều độ sâu khởi tạo từ nhỏ đến lớn. Kết hợp cả hai cách
cho quần thể ban đầu đa dạng hình dạng hơn hẳn so với chỉ dùng một
cách.

In [4]:
def random_terminal(rng):
    if rng.random() < 0.5:
        return Node("x")
    return Node(round(float(rng.uniform(-2, 2)), 3))


def random_tree(rng, depth_remaining, method):
    la_la = depth_remaining <= 1 or (method == "grow" and rng.random() < 0.3)
    if la_la:
        return random_terminal(rng)
    op = rng.choice(list(FUNCTIONS_ARITY.keys()))
    arity = FUNCTIONS_ARITY[op]
    return Node(op, [random_tree(rng, depth_remaining - 1, method) for _ in range(arity)])


def init_population(rng, population_size, min_depth, max_depth):
    population = []
    do_sau_list = list(range(min_depth, max_depth + 1))
    for i in range(population_size):
        do_sau = do_sau_list[i % len(do_sau_list)]
        method = "full" if i % 2 == 0 else "grow"
        population.append(random_tree(rng, do_sau, method))
    return population

---
##### Toán tử di truyền trên cây — lai ghép & đột biến

- **Lai ghép (subtree crossover)**: chọn ngẫu nhiên một nút ở mỗi cây
  cha mẹ, HOÁN ĐỔI hai cây con tại đó cho nhau.
- **Đột biến (subtree mutation)**: chọn ngẫu nhiên một nút, THAY bằng
  một cây con mới sinh ngẫu nhiên.

Cả hai đều bị giới hạn bởi `max_depth` — nếu kết quả vượt quá độ sâu
cho phép thì hủy thao tác, giữ nguyên cá thể gốc. Đây là cách kiểm soát
**bloat** (cây phình to vô tội vạ qua các thế hệ) đơn giản nhất trong
GP.

In [5]:
def crossover(rng, parent1, parent2, max_depth):
    child1 = copy_tree(parent1)
    child2 = copy_tree(parent2)

    nodes1 = all_nodes(child1)
    nodes2 = all_nodes(child2)
    diem1 = nodes1[rng.integers(0, len(nodes1))]
    diem2 = nodes2[rng.integers(0, len(nodes2))]

    # Hoán đổi cây con tại 2 điểm đã chọn — vì diem1/diem2 là tham
    # chiếu TRỰC TIẾP tới nút bên trong child1/child2, đổi op/children
    # của chúng tương đương đổi cả nhánh cây tại đó.
    diem1.op, diem2.op = diem2.op, diem1.op
    diem1.children, diem2.children = diem2.children, diem1.children

    if tree_depth(child1) > max_depth or tree_depth(child2) > max_depth:
        return copy_tree(parent1), copy_tree(parent2)

    return child1, child2


def mutate(rng, individual, max_depth):
    child = copy_tree(individual)
    nodes = all_nodes(child)
    diem = nodes[rng.integers(0, len(nodes))]

    cay_moi = random_tree(rng, int(rng.integers(1, 4)), rng.choice(["grow", "full"]))
    diem.op = cay_moi.op
    diem.children = cay_moi.children

    if tree_depth(child) > max_depth:
        return copy_tree(individual)

    return child


def tournament_selection(rng, population, fitnesses, tournament_size):
    indices = rng.integers(0, len(population), size=tournament_size)
    best_index = indices[np.argmin(fitnesses[indices])]
    return population[best_index]

---
##### Hàm thích nghi (fitness) — MSE + áp lực đơn giản hóa

Fitness = MSE giữa cây dự đoán và dữ liệu, CỘNG THÊM một khoản phạt tỉ
lệ với kích thước cây (`PARSIMONY_COEF × tree_size`). Không có khoản
phạt này, GP có xu hướng "bloat" — cây ngày càng to, phức tạp hơn mức
cần thiết mà không cải thiện được độ khớp bao nhiêu (một hiện tượng
kinh điển của GP). Cây cho giá trị không hữu hạn (chia cho 0 vượt khỏi
protected_div, hoặc tràn số) bị phạt fitness = $\infty$.

In [6]:
PARSIMONY_COEF = 0.0005


def fitness(node, X, y):
    y_pred = eval_node(node, X)
    y_pred = np.broadcast_to(np.asarray(y_pred, dtype=float), X.shape)

    if not np.all(np.isfinite(y_pred)):
        return np.inf

    mse = np.mean((y_pred - y) ** 2)

    return mse + PARSIMONY_COEF * tree_size(node)

---
##### Vòng lặp tiến hóa (Genetic Programming)

Cấu trúc giống hệt `genetic_algorithm_unconstrained` ở
`01_unconstrained_optimization.ipynb` (elitism + tournament selection +
lai ghép + đột biến qua từng thế hệ) — chỉ khác biểu diễn cá thể là
CÂY thay vì VECTOR số thực.

In [7]:
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.15
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3


def genetic_programming(
    X, y,
    population_size,
    generations,
    max_depth=6,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    seed=42,
):
    """GP tìm cây biểu thức khớp nhất với dữ liệu (X, y), tối thiểu hóa
    MSE + áp lực đơn giản hóa."""

    rng = np.random.default_rng(seed)

    population = init_population(rng, population_size, min_depth=2, max_depth=max_depth)
    fitnesses = np.array([fitness(ind, X, y) for ind in population])

    best_index = int(np.argmin(fitnesses))
    best_tree = copy_tree(population[best_index])
    best_fitness = fitnesses[best_index]

    history = []

    start_time = time.perf_counter()

    for generation in range(generations):

        current_best = int(np.argmin(fitnesses))
        if fitnesses[current_best] < best_fitness:
            best_fitness = fitnesses[current_best]
            best_tree = copy_tree(population[current_best])

        history.append(best_fitness)

        order = np.argsort(fitnesses)
        new_population = [copy_tree(population[i]) for i in order[:elite_size]]

        while len(new_population) < population_size:
            parent1 = tournament_selection(rng, population, fitnesses, tournament_size)
            parent2 = tournament_selection(rng, population, fitnesses, tournament_size)

            if rng.random() < crossover_rate:
                child1, child2 = crossover(rng, parent1, parent2, max_depth)
            else:
                child1, child2 = copy_tree(parent1), copy_tree(parent2)

            if rng.random() < mutation_rate:
                child1 = mutate(rng, child1, max_depth)
            if rng.random() < mutation_rate:
                child2 = mutate(rng, child2, max_depth)

            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population
        fitnesses = np.array([fitness(ind, X, y) for ind in population])

    current_best = int(np.argmin(fitnesses))
    if fitnesses[current_best] < best_fitness:
        best_fitness = fitnesses[current_best]
        best_tree = copy_tree(population[current_best])

    elapsed_time = time.perf_counter() - start_time

    generations_run = int(np.argmin(history)) + 1

    return {
        "tree": best_tree,
        "fitness": best_fitness,
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }

---
##### Chạy GP

In [8]:
GP_POPULATION_SIZE = 300
GP_GENERATIONS = 60
GP_MAX_DEPTH = 6

# Mỗi lần đánh giá chỉ là tính f(x) trên 60 điểm bằng NumPy thuần — rẻ
# hơn hẳn so với việc huấn luyện một mô hình ML thật (khác hẳn notebook
# 04 cũ) — nên có thể dùng quần thể/số thế hệ lớn như GA số thực ở 01.
gp_result = genetic_programming(
    X_train, y_train,
    population_size=GP_POPULATION_SIZE,
    generations=GP_GENERATIONS,
    max_depth=GP_MAX_DEPTH,
    seed=42,
)

print(f"Thời gian chạy GP: {gp_result['time']:.2f}s")
print(f"Fitness tốt nhất (MSE + phạt kích thước): {gp_result['fitness']:.6f}")
print(f"Đạt được từ thế hệ: {gp_result['generations_run']}/{gp_result['generations']}")
print(f"Kích thước cây: {tree_size(gp_result['tree'])} nút, độ sâu {tree_depth(gp_result['tree'])}")

Thời gian chạy GP: 4.79s
Fitness tốt nhất (MSE + phạt kích thước): 0.009023
Đạt được từ thế hệ: 30/60
Kích thước cây: 6 nút, độ sâu 3


---
##### Biểu thức GP tìm được (rút gọn qua SymPy)

Chuyển cây sang biểu thức SymPy rồi gọi `sympy.simplify()` — vừa rút
gọn các hằng số/toán tử dư thừa GP sinh ra trong quá trình tiến hóa,
vừa cho phép hiển thị công thức dưới dạng LaTeX dễ đọc.

In [9]:
X_SYM = sp.symbols("x")


def tree_to_sympy(node):
    if node.op == "x":
        return X_SYM
    if isinstance(node.op, (int, float)):
        return sp.Float(node.op)
    args = [tree_to_sympy(con) for con in node.children]
    if node.op == "+":
        return args[0] + args[1]
    if node.op == "-":
        return args[0] - args[1]
    if node.op == "*":
        return args[0] * args[1]
    if node.op == "/":
        return args[0] / args[1]
    if node.op == "sin":
        return sp.sin(args[0])
    if node.op == "cos":
        return sp.cos(args[0])
    raise ValueError(f"Toán tử không xác định: {node.op}")


bieu_thuc_gp = sp.nsimplify(sp.simplify(tree_to_sympy(gp_result["tree"])), tolerance=1e-3, rational=False)

display(Markdown(
    "**Biểu thức GP tìm được:**\n\n$$f_{GP}(x) = " + sp.latex(bieu_thuc_gp) + "$$"
))

**Biểu thức GP tìm được:**

$$f_{GP}(x) = x^{2} + \sin{\left(x \right)}$$

---
##### Đánh giá trên tập test — so với hàm ẩn thật

Sinh một tập test MỚI (không nhiễu) trực tiếp từ `ham_an` — chấm điểm
khách quan xem GP có thực sự tìm lại được TÍN HIỆU thật hay chỉ khớp
với nhiễu của tập train.

In [10]:
N_TEST = 200

X_test = rng.uniform(-3, 3, size=N_TEST)
y_test_true = ham_an(X_test)

y_pred_gp = eval_node(gp_result["tree"], X_test)
y_pred_gp = np.broadcast_to(np.asarray(y_pred_gp, dtype=float), X_test.shape)

rmse_gp = np.sqrt(np.mean((y_pred_gp - y_test_true) ** 2))

print(f"RMSE trên tập test (so với hàm ẩn thật, không nhiễu): {rmse_gp:.6f}")

RMSE trên tập test (so với hàm ẩn thật, không nhiễu): 0.000000


---
##### So sánh với hồi quy đa thức (không biết trước dạng hàm)

Nếu không dùng GP, một cách tiếp cận thông thường là: đoán một dạng mô
hình (ví dụ đa thức bậc 4) rồi fit tham số bằng bình phương tối thiểu
(`numpy.polyfit`) — nhưng phải CHỌN TRƯỚC bậc đa thức, và một đa thức
thuần túy không thể biểu diễn chính xác thành phần $\sin(x)$ trong hàm
ẩn, dù bậc có cao đến đâu.

In [11]:
POLY_DEGREE = 4

he_so = np.polyfit(X_train, y_train, POLY_DEGREE)

y_pred_poly = np.polyval(he_so, X_test)
rmse_poly = np.sqrt(np.mean((y_pred_poly - y_test_true) ** 2))

bieu_thuc_poly = sp.Poly([round(float(c), 4) for c in he_so], X_SYM).as_expr()

display(Markdown(
    "**Biểu thức hồi quy đa thức (bậc " + str(POLY_DEGREE) + "):**\n\n$$f_{poly}(x) = "
    + sp.latex(bieu_thuc_poly) + "$$"
))
print(f"\nRMSE trên tập test (so với hàm ẩn thật, không nhiễu): {rmse_poly:.6f}")

**Biểu thức hồi quy đa thức (bậc 4):**

$$f_{poly}(x) = - 0.0016 x^{4} - 0.0941 x^{3} + 1.0154 x^{2} + 0.8522 x - 0.0387$$


RMSE trên tập test (so với hàm ẩn thật, không nhiễu): 0.058908


---
##### Bảng so sánh

In [12]:
display(Markdown(
    "| Phương pháp | Biểu thức | RMSE (test, so với hàm ẩn thật) |\n"
    "|---|---|---|\n"
    f"| Hồi quy đa thức (bậc {POLY_DEGREE}) | ${sp.latex(bieu_thuc_poly)}$ | ${rmse_poly:.6f}$ |\n"
    f"| GP (Symbolic Regression) | ${sp.latex(bieu_thuc_gp)}$ | ${rmse_gp:.6f}$ |"
))

| Phương pháp | Biểu thức | RMSE (test, so với hàm ẩn thật) |
|---|---|---|
| Hồi quy đa thức (bậc 4) | $- 0.0016 x^{4} - 0.0941 x^{3} + 1.0154 x^{2} + 0.8522 x - 0.0387$ | $0.058908$ |
| GP (Symbolic Regression) | $x^{2} + \sin{\left(x \right)}$ | $0.000000$ |